<a href="https://colab.research.google.com/github/mohamed468/arabic-english-semantic-chunker/blob/main/notebooks/arabic_english_semantic_chunking_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Arabic-English Semantic Chunker: From Training to Production

## Introduction

In this project, I built a complete end-to-end pipeline for fine-tuning a large language model on **Semantic Text Chunking** for Arabic, English, and mixed-language (Code-Switched) content. I used **Qwen 2.5 1.5B Instruct** as the base model, fine-tuned it using **Unsloth** and **LoRA**, deployed it for production inference using **vLLM**, and stress-tested it under load using **Locust**.

## What I Learned & Built

By the end of this project, I covered and implemented:

- **Environment Setup:** Installing and configuring required libraries (`transformers`, `vllm`, `peft`), including resolving real-world version conflicts between numpy, CUDA, and the pip dependency resolver.
- **Output Schema Design:** Designing a strict Pydantic schema to ensure the model produces structured, validated JSON output (`original_text_length`, `semantic_chunks`).
- **Parameter-Efficient Fine-Tuning:** Accelerating the training process via **Unsloth** to fit the chunking task using a LoRA adapter effectively on limited VRAM.
- **Post-Training Evaluation & Sanitization:** Building a custom `Generator` class with a `logits_processor` to suppress unwanted Chinese-character generation during standard inference.
- **Cost Estimation:** Measuring actual input/output token consumption on real samples to estimate operational distillation costs.
- **vLLM Deployment:** Running a high-performance, OpenAI-compatible inference server with optimized PagedAttention and dynamic LoRA support.
- **Progressive Load Testing:** Building a Locust script to simulate concurrent users (up to 20 users) and measure real production throughput (tokens/sec) on a T4 GPU under stress.

## Who Should Explore This

This project is aimed at:

- **ML Engineers** interested in a complete LLMOps pipeline from training to deployment.
- **NLP Practitioners** working specifically with Arabic and multilingual text segmentation.
- **RAG System Builders** looking for a smarter alternative to naive character/sentence chunking to preserve contextual embeddings.
- **Developers** interested in deploying and benchmarking fine-tuned models via vLLM.

---

This project follows a **practical workflow** for fine-tuning LLMs with real-world examples, covering the full journey from data schema design to production deployment and load testing.

**Author:**
- [Muhammad Abd El-Fattah](https://www.linkedin.com/in/mohamed456/) | [GitHub Profile](https://github.com/mohamed468)

# Mount Your Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -qU "numpy<2" json-repair==0.29.1 faker==35.2.0 pydantic google-generativeai tqdm wandb requests datasets==3.2.0 transformers==4.48.3

# Setup

In [ ]:
from google.colab import userdata
import wandb

wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('huggingface')
!huggingface-cli login --token {hf_token}

# Imports

In [ ]:
import requests
from typing import List, Optional, Literal
from datetime import datetime

import json_repair
import torch
import json, os, random, time
from tqdm.auto import tqdm
from os.path import join
from pydantic import BaseModel, Field
import google.generativeai as genai


data_dir = "/content/drive/MyDrive/semantic-chunking-project"
datasets_dir = join(data_dir, "datasets")
os.makedirs(datasets_dir, exist_ok=True)
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = None

def parse_json(text):
  try:
    return json_repair.loads(text)
  except:
    return None

# Task

In [ ]:
arabic_text_sample = """
ملف بيانات تاريخي: الصراع بين نيكولا تسلا وتوماس إديسون (حرب التيارات)
1. نبذة عن توماس إديسون (Thomas Edison)
أسلوب العمل: كان إديسون رجل أعمال ومخترعاً يعتمد على التجربة والخطأ (Trial and Error). كان يمتلك فريقاً كبيراً من المساعدين في مختبره الشهير في "مينلو بارك".

أبرز إنجازاته: تطوير المصباح الكهربائي العملي، الفونوغراف (جهاز تسجيل الصوت)، وكاميرا الصور المتحركة.

رؤيته للكهرباء: كان الداعم الرئيسي للتيار المستمر (Direct Current - DC)، وهو نظام ينقل الكهرباء في اتجاه واحد، لكنه كان يعاني من مشكلة قصر المسافة، حيث كان يتطلب بناء محطة توليد كل ميل تقريباً.

2. نبذة عن نيكولا تسلا (Nikola Tesla)
أسلوب العمل: كان تسلا مهندساً وعالماً يعتمد على التخيل البصري والحسابات الذهنية المعقدة. كان يبني النماذج في عقله ويختبرها ذهنياً قبل أن يصنعها على أرض الواقع.

أبرز إنجازاته: ابتكار المحرك الحثي الذي يعمل بالتيار المتردد، وملف تسلا (Tesla Coil)، والإسهام في تطوير تكنولوجيا الراديو.

رؤيته للكهرباء: كان الداعم الرئيسي للتيار المتردد (Alternating Current - AC)، وهو نظام يسمح بنقل الكهرباء لمسافات طويلة جداً وتغيير الجهد الكهربائي باستخدام المحولات.

3. بداية العمل المشترك والخلاف الأول (قصة المكافأة الوهمية)
اللقاء الأول: هاجر تسلا إلى الولايات المتحدة عام 1884 وبدأ العمل في شركة إديسون. أُعجب إديسون بذكاء تسلا وكلفه بتطوير وتحسين مولدات التيار المستمر الخاصة بالشركة.

قصة الـ 50 ألف دولار: عرض إديسون على تسلا مكافأة قدرها 50,000 دولار (مبلغ ضخم جداً في ذلك الوقت) إذا تمكن من إصلاح وتطوير المولدات. بعد أشهر من العمل الشاق، نجح تسلا في المهمة وطلب المكافأة.

رد إديسون والاستقالة: ضحك إديسون وقال مقولته الشهيرة: "يا تسلا، أنت لا تفهم حس الدعابة الأمريكي". غضب تسلا بشدة من هذا الموقف، واستقال من الشركة فوراً، ليتحول من موظف إلى أشرس منافس لإديسون.

4. حرب التيارات (War of the Currents)
جوهر الصراع: تحالف تسلا مع رجل الأعمال "جورج ويستينغهاوس" لنشر نظام التيار المتردد (AC)، بينما استمر إديسون في الدفاع عن نظام التيار المستمر (DC) الذي استثمر فيه أموالاً طائلة.

حملات التشويه: لإثبات أن تيار تسلا المتردد خطير وقاتل، أطلق إديسون حملة علاقات عامة لتشويه السمعة. قام بصعق حيوانات (مثل الكلاب والخيول وحتى فيل يُدعى توبسي) علناً باستخدام التيار المتردد.

الكرسي الكهربائي: موّل إديسون سراً تطوير أول كرسي إعدام كهربائي يعمل بالتيار المتردد، محاولاً ربط اسم تيار تسلا بالموت في أذهان العامة.
"""

# chunking_message

In [ ]:
class SemanticChunking(BaseModel):
  original_text_length: int = Field(..., description="The total number of characters in the original text.")
  semantic_chunks: List[str] = Field(
        ...,
        min_length=2,
        description="List of sequential chunks from the original text. "
            "Each chunk is self-contained and preserves the original language(s). "
            "Arabic stays Arabic, English stays English, mixed stays mixed."
    )

chunking_messages = [
    {
        "role": "system",
        "content": "\n".join([
            "You are an expert multilingual NLP assistant specializing in Semantic Chunking.",
            "You process Arabic, English, and mixed Arabic-English text with equal proficiency.",
            "Split the provided text into meaningful, self-contained semantic chunks.",
            "Preserve ALL original languages exactly — do not translate, summarize, or paraphrase.",
            "For mixed text: keep each chunk coherent even if it contains both languages.",
            "You MUST output a valid JSON perfectly matching the provided Pydantic schema.",
            "Do not generate any introduction or conclusion."
        ])
    },
    {
        "role": "user",
        "content": "\n".join([
            "## Arabic Text:",
            arabic_text_sample.strip(),
            "",
            "## Pydantic Schema:",
            json.dumps(SemanticChunking.model_json_schema(), ensure_ascii=False),
            "",
            "## JSON Output:",
            "```json\n"
        ])
    }
]

# Baseline Evaluation: Local Model (Qwen 1.5B)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)

In [ ]:
model

In [ ]:
text = tokenizer.apply_chat_template(
    chunking_messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)
generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n=== Qwen 1.5B Baseline  Output ===")
print(response)

parsed_response = parse_json(response)
print("\n=== Parsed JSON ===")
print(parsed_response)

# Evaluate Gemini

In [ ]:
import google.generativeai as genai
from google.colab import userdata

API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)
gemini_test_model = genai.GenerativeModel('gemini-3.1-flash-lite')
response = gemini_test_model.generate_content("What is the capital of Egypt? Answer in one word.")
print(response.text)


In [ ]:
teacher_model = genai.GenerativeModel(
    model_name='gemini-3.1-flash-lite',
    system_instruction=chunking_messages[0]['content']
)


chat_completion = teacher_model.generate_content(
    chunking_messages[1]['content'],
    generation_config=genai.GenerationConfig(
        temperature=0.2,
        response_mime_type="application/json"
    )
)

print("=== gemini-3.1-flash-lite Output ===")
print(chat_completion.text)

parsed_teacher_response = parse_json(chat_completion.text)
print("\n=== Parsed JSON (Teacher Quality) ===")
print(parsed_teacher_response)

# Data Preparation

In [ ]:
# ============================================================
# Data Preparation
# ============================================================
# Merges 3 sources into raw_arabic_combined.jsonl
# Already executed — this block is kept as an archive only, no need to re-run
# ============================================================
# import tarfile, json, random, os
# from datasets import load_dataset
# from os.path import join
# data_dir      = "/content/drive/MyDrive/semantic-chunking-project"
# datasets_dir  = join(data_dir, "datasets")
# xlsum_bz2     = join(datasets_dir, "arabic_XLSum_v2.0.tar.bz2")
# save_path     = join(datasets_dir, "raw_arabic_combined.jsonl")
#
# all_data = []
#
# # Source 1: Wikipedia (formal standard Arabic)
# wiki = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train[:1000]")
# saved_wiki = 0
# for ix, item in enumerate(wiki):
#     if saved_wiki >= 526:
#         break
#     text = item["text"][:1500].strip()
#     if len(text) >= 200:
#         all_data.append({"id": f"wiki_{ix}", "content": text})
#         saved_wiki += 1
# print(f"saved_wiki: {saved_wiki}")
#
# # Source 2: XLSum (BBC news articles)
# with tarfile.open(xlsum_bz2, "r:bz2") as tar:
#     tar.extractall("/tmp/xlsum/")
#
# xlsum_raw = []
# with open("/tmp/xlsum/arabic_train.jsonl", encoding="utf-8") as f:
#     for line in f:
#         if line.strip():
#             xlsum_raw.append(json.loads(line))
#
# for i, item in enumerate(xlsum_raw[:1500]):
#     text = item["text"][:1500].strip()
#     if len(text) >= 200:
#         all_data.append({"id": f"xlsum_{i}", "content": text})
#
# # Source 3: technical Wikipedia articles with mixed Arabic/English content
# wiki_tech = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train[1500:6000]")
# added_mixed = 0
# for ix, item in enumerate(wiki_tech):
#     if added_mixed >= 500:
#         break
#
#     text = item["text"][:1500].strip()
#
#     # Compute the English ratio — only accept genuinely mixed text (5% to 40%)
#     english_chars = sum(1 for c in text if c.isascii() and c.isalpha())
#     total_chars   = sum(1 for c in text if c.isalpha())
#     ratio = english_chars / total_chars if total_chars > 0 else 0
#
#     if 0.05 <= ratio <= 0.40 and len(text) >= 200:
#         all_data.append({"id": f"mixed_{ix}", "content": text})
#         added_mixed += 1
# print(f"added_mixed: {added_mixed}")
#
# # Shuffle with a fixed seed and save the final combined file
# random.Random(101).shuffle(all_data)
# with open(save_path, "w", encoding="utf-8") as f:
#     for item in all_data:
#         f.write(json.dumps(item, ensure_ascii=False) + "\n")
# print(f"Saved {len(all_data)} articles -> {save_path}")

# Knowledge Distillation (Building SFT Dataset)

In [ ]:
import json, random
from os.path import join

data_dir = "/content/drive/MyDrive/semantic-chunking-project"
datasets_dir = join(data_dir, "datasets")
raw_data_path = join(datasets_dir, "raw_arabic_combined.jsonl")

raw_data = []
for line in open(raw_data_path, encoding="utf-8"):
    if line.strip():
        raw_data.append(json.loads(line.strip()))
random.Random(101).shuffle(raw_data)

wiki_count  = sum(1 for d in raw_data if d["id"].startswith("wiki_"))
news_count  = sum(1 for d in raw_data if d["id"].startswith("xlsum_"))
mixed_count = sum(1 for d in raw_data if d["id"].startswith("mixed_"))

print(f"Total raw articles ready for distillation: {len(raw_data)}")
print(f"  Wikipedia : {wiki_count}")
print(f"  XLSum     : {news_count}")
print(f"  Mixed     : {mixed_count}")

In [ ]:
MODEL_NAME  = "gemini-3.1-flash-lite"
PRICE_IN    = 0.25 / 1e6
PRICE_OUT   = 1.50 / 1e6
SLEEP_SEC   = 5
SAVE_TO     = join(datasets_dir, "sft_chunking.jsonl")
schema_str  = json.dumps(SemanticChunking.model_json_schema(), ensure_ascii=False)

print(f"Output file: {SAVE_TO}")

In [ ]:
# ============================================================
# Distillation loop — rotates across multiple API keys
# automatically when the daily request limit is reached
# ============================================================
import os, json, time
import google.generativeai as genai
from os.path import join
from google.colab import userdata

API_KEYS = [
    userdata.get('GEMINI_API_KEY_1'),
    userdata.get('GEMINI_API_KEY_2'),
    userdata.get('GEMINI_API_KEY_3'),
    userdata.get('GEMINI_API_KEY_4'),
]
API_KEYS = [k for k in API_KEYS if k]

if not API_KEYS:
    raise ValueError("No valid API key found in Colab Secrets.")

print(f"Available API keys: {len(API_KEYS)}")

current_key_ix = 0

def get_teacher_model(key_ix):
    genai.configure(api_key=API_KEYS[key_ix])
    return genai.GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=chunking_messages[0]["content"]
    )

teacher_model = get_teacher_model(current_key_ix)
print(f"Starting with API key #{current_key_ix + 1}")

done_ids = set()
if os.path.exists(SAVE_TO):
    for line in open(SAVE_TO, encoding="utf-8"):
        if line.strip():
            done_ids.add(json.loads(line)["id"])

remaining = [d for d in raw_data if d["id"] not in done_ids]
print(f"Total: {len(raw_data)} | Done: {len(done_ids)} | Remaining: {len(remaining)}\n")

prompt_tokens = completion_tokens = successful_generations = 0

ix = 0
while ix < len(remaining):
    data_item = remaining[ix]
    current_text = data_item["content"].strip()

    user_prompt = "\n".join([
        "## Input Text:\n", current_text, "\n",
        "## Pydantic Schema:\n", schema_str, "\n",
        "## JSON Output:\n", "```json\n"
    ])

    try:
        response = teacher_model.generate_content(
            user_prompt,
            generation_config=genai.GenerationConfig(
                temperature=0.2,
                response_mime_type="application/json"
            )
        )

        if response.candidates[0].finish_reason.name != "STOP":
            ix += 1
            continue

        llm_resp_dict = parse_json(response.text)
        if not llm_resp_dict:
            ix += 1
            continue

        prompt_tokens += response.usage_metadata.prompt_token_count
        completion_tokens += response.usage_metadata.candidates_token_count

        with open(SAVE_TO, "a", encoding="utf-8") as dest:
            dest.write(json.dumps({
                "id": data_item["id"],
                "original_text": current_text,
                "task": "Split the provided text into meaningful, self-contained semantic chunks. Do NOT summarize or leave out words.",
                "output_schema": schema_str,
                "teacher_response": llm_resp_dict,
            }, ensure_ascii=False, default=str) + "\n")

        successful_generations += 1
        ix += 1

        if successful_generations % 10 == 0:
            cost = (prompt_tokens * PRICE_IN) + (completion_tokens * PRICE_OUT)
            print(f"[{ix}/{len(remaining)}] Done: {successful_generations} | Cost: ${cost:.5f}")

        time.sleep(SLEEP_SEC)

    except Exception as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            current_key_ix += 1
            if current_key_ix >= len(API_KEYS):
                print(f"\nAll {len(API_KEYS)} API keys exhausted.")
                print(f"{successful_generations} articles processed successfully.")
                print("Re-run this cell later — it will resume automatically from where it stopped.")
                break
            print(f"\nAPI key #{current_key_ix} exhausted. Switching to key #{current_key_ix + 1}...")
            teacher_model = get_teacher_model(current_key_ix)
            continue
        print(f"Unexpected error: {e}")
        ix += 1
        continue

cost = (prompt_tokens * PRICE_IN) + (completion_tokens * PRICE_OUT)
print(f"\nSession finished: {successful_generations} new articles saved | Cost: ${cost:.5f}")

In [ ]:
# ============================================================
# Progress check to see how many articles are done so far
# ============================================================
import json, os

all_records = []
if os.path.exists(SAVE_TO):
    for line in open(SAVE_TO, encoding="utf-8"):
        if line.strip():
            all_records.append(json.loads(line.strip()))

wiki_done  = sum(1 for r in all_records if r["id"].startswith("wiki_"))
xlsum_done = sum(1 for r in all_records if r["id"].startswith("xlsum_"))
mixed_done = sum(1 for r in all_records if r["id"].startswith("mixed_"))

print("Distilled SFT data completed so far:")
print(f"  Wikipedia : {wiki_done}")
print(f"  XLSum     : {xlsum_done}")
print(f"  Mixed     : {mixed_done}")
print(f"  Total     : {len(all_records)}")

# Format Finetuning Datasets

In [ ]:
# ============================================================
# Format Finetuning Datasets
# ============================================================
import json
import random
from os.path import join
from datasets import Dataset

sft_data_path = join(datasets_dir, "sft_chunking.jsonl")

system_message = "\n".join([
    "You are a professional multilingual NLP data parser.",
    "Split the provided text into meaningful, self-contained semantic chunks.",
    "Follow the Task and Output Schema to generate the Output JSON.",
    "Do not add any introduction or conclusion.",
    "Preserve the original text exactly and respect its language — do not summarize, translate, or paraphrase."
])

llm_finetuning_data = []

for line in open(sft_data_path, encoding="utf-8"):
    if line.strip() == "":
        continue
    rec = json.loads(line.strip())

    llm_finetuning_data.append({
        "system": system_message,
        "instruction": "\n".join([
            "# Input Text:",
            rec["original_text"],
            "",
            "# Task:",
            rec["task"],
            "",
            "# Output Schema:",
            rec["output_schema"],
            "",
            "# Output JSON:",
            "```json\n"
        ]),
        "input": "",
        "output": json.dumps(rec["teacher_response"], ensure_ascii=False, default=str) + "\n```",
        "history": []
    })

random.Random(101).shuffle(llm_finetuning_data)
hf_dataset = Dataset.from_list(llm_finetuning_data)

print(f"Total formatted examples: {len(hf_dataset)}")
print("\n--- Preview ---")
ex = hf_dataset[0]
print(f"Instruction[:100]: {ex['instruction'][:100]}...")
print(f"Output[:100]:      {ex['output'][:100]}...")

In [ ]:
len(llm_finetuning_data)

In [ ]:
# ============================================================
# Train / Val / Test split
# ============================================================
import os

train_sz = 1750
val_sz   = 128

total_size = len(llm_finetuning_data)
assert train_sz + val_sz < total_size, (
    f"train_sz + val_sz ({train_sz + val_sz}) must be smaller than the "
    f"total dataset size ({total_size}); adjust the split sizes."
)

train_ds = llm_finetuning_data[:train_sz]
val_ds   = llm_finetuning_data[train_sz : train_sz + val_sz]
test_ds  = llm_finetuning_data[train_sz + val_sz:]

print(f"Train : {len(train_ds)}")
print(f"Val   : {len(val_ds)}")
print(f"Test  : {len(test_ds)}")
print(f"Total : {len(train_ds) + len(val_ds) + len(test_ds)}")

splits_dir = join(datasets_dir, "unsloth-finetune-data")
os.makedirs(splits_dir, exist_ok=True)

for name, data in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    with open(join(splits_dir, f"{name}.json"), "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, default=str, indent=2)

print(f"Saved to {splits_dir}")

train_dataset = Dataset.from_list(train_ds)
val_dataset   = Dataset.from_list(val_ds)
test_dataset  = Dataset.from_list(test_ds)

🔴 **Restart Runtime required here** 🔴

Installing Unsloth may upgrade numpy or other C-extension packages
that are already loaded in memory from Stage 1. Restart before
proceeding to avoid a repeat of the numpy mid-session upgrade error.

After restarting, re-run only the Drive mount + imports cells,
then proceed directly to the Unsloth install cell below.

# Finetuning

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q wandb

In [ ]:
# ════════════════════════════════════════════════
# Load base model via Unsloth (required before get_peft_model)
# ════════════════════════════════════════════════
from unsloth import FastLanguageModel

base_model_id = "unsloth/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = base_model_id,
    max_seq_length = 3500,
    dtype          = None,
    load_in_4bit   = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA ready")
model.print_trainable_parameters()

In [ ]:
def format_example(example):
    messages = [
        {"role": "system",    "content": example["system"]},
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = train_dataset.map(format_example)
val_dataset   = val_dataset.map(format_example)

print(f"Train dataset ready: {len(train_dataset)} examples")
print(f"Val dataset ready  : {len(val_dataset)} examples")

In [ ]:
# ============================================================
# Train (SFTTrainer)
# ============================================================
import wandb
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

output_dir = join(data_dir, "models")

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    dataset_text_field = "text",
    max_seq_length  = 3500,
    args = SFTConfig(
        per_device_train_batch_size  = 1,
        gradient_accumulation_steps  = 4,
        learning_rate                = 1e-4,
        lr_scheduler_type            = "cosine",
        warmup_ratio                 = 0.1,
        num_train_epochs             = 3,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps                = 10,
        save_steps                   = 500,
        eval_strategy                = "steps",
        eval_steps                   = 100,
        output_dir                   = output_dir,
        optim                        = "adamw_8bit",
        seed                         = 42,
        report_to                    = "wandb",
        run_name                     = "semantic-chunking-qwen-1.5b",
    ),
)

print("Training started")
trainer.train()
print("Training finished")

# New Finetuned Model Evaluation

In [ ]:
from unsloth import FastLanguageModel

base_model_id = "unsloth/Qwen2.5-1.5B-Instruct"
finetuned_model_id = join(data_dir, "models/checkpoint-1000")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = finetuned_model_id,
    max_seq_length = 3500,
    dtype = torch.float16,
    load_in_4bit = False,
)

FastLanguageModel.for_inference(model)

def generate_resp(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,
    )
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

response = generate_resp(chunking_messages)

print("=== Raw Output ===")
print(response)
print("\n=== Parsed JSON ===")
print(parse_json(response))

# Tip for Qwen2.5
Qwen2.5 oftenly produce chinese characters with some responses. To skip this, use the next class to generate responses.

Source: https://jupyter267.medium.com/how-to-eliminate-the-chance-of-generating-chinese-in-qwen-2-5-2cf919bb0fdc

In [ ]:
class Generator:
    def __init__(self, model, tokenizer):
        self.model, self.tokenizer = model, tokenizer
        self.mask = None

    def generate(self, messages: list, max_new_tokens: int = 2000):
        def logits_processor(token_ids, logits):
            if self.mask is None:
                token_ids_vocab = torch.arange(logits.size(-1))
                decoded_tokens = self.tokenizer.batch_decode(token_ids_vocab.unsqueeze(1), skip_special_tokens=True)
                self.mask = torch.tensor([
                    any(0x4E00 <= ord(c) <= 0x9FFF or 0x3400 <= ord(c) <= 0x4DBF or 0xF900 <= ord(c) <= 0xFAFF for c in token)
                    for token in decoded_tokens
                ])
            mask_on_device = self.mask.to(logits.device)
            logits[:, mask_on_device] = -float("inf")
            return logits

        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_k=None,
            top_p=None,
            logits_processor=[logits_processor]
        )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        return self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

llm = Generator(model, tokenizer)
response = llm.generate(chunking_messages)
print(parse_json(response))

In [ ]:
# ============================================================
# Push Fine-tuned Model to HuggingFace Hub
# ============================================================
from unsloth import FastLanguageModel
from os.path import join

HF_USERNAME = "Mo-Abdelfattah"
REPO_NAME   = "arabic-semantic-chunker-qwen1.5b"
REPO_ID     = f"{HF_USERNAME}/{REPO_NAME}"

model.push_to_hub(REPO_ID, private=False)
tokenizer.push_to_hub(REPO_ID, private=False)

print(f"Model pushed to: https://huggingface.co/{REPO_ID}")

# Cost Estimation

In [ ]:
from tqdm.auto import tqdm
from faker import Faker
from datetime import datetime

start_time = datetime.now()
fake = Faker('ar')
input_tokens = 0
output_tokens = 0

system_message = chunking_messages[0]["content"]
schema_str = json.dumps(SemanticChunking.model_json_schema(), ensure_ascii=False)

for i in tqdm(range(30)):
    prompt_text = fake.text(max_nb_chars=random.randint(150, 200))
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": "\n".join([
            "# Input Text:", prompt_text, "",
            "# Task:", "Split the provided text into meaningful, self-contained semantic chunks. Do NOT summarize or leave out words.", "",
            "# Output Schema:", schema_str, "",
            "# Output JSON:", "```json\n"
        ])}
    ]
    response = llm.generate(messages)

    rendered_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_tokens  += len(tokenizer.encode(rendered_prompt))
    output_tokens += len(tokenizer.encode(response))

total_time = (datetime.now() - start_time).total_seconds()
print(f"Total Time: {total_time} seconds")
print(f"Input Tokens: {input_tokens}")
print(f"Output Tokens: {output_tokens}")
print(f"Total Tokens: {input_tokens + output_tokens}")

# vLLM

In [ ]:
!pip install -qU "numpy<2" vllm==0.14.0

🛑 **Restart Runtime required here** 🛑

The in-memory model and tokenizer from the previous sections must be cleared
before starting the vLLM server, otherwise GPU memory conflicts may occur.

After restarting, only re-run the Drive mount + imports cells from Part 1 —
do NOT re-run the training or evaluation cells.

In [ ]:
import os
import time
import requests
from os.path import join

data_dir = "/content/drive/MyDrive/semantic-chunking-project"

base_model_id    = "unsloth/Qwen2.5-1.5B-Instruct"
adapter_model_id = join(data_dir, "models/checkpoint-1000")

os.system("pkill -9 -f vllm")
time.sleep(3)  # brief pause to ensure memory is released before restarting

!nohup vllm serve "{base_model_id}" \
  --dtype=half \
  --gpu-memory-utilization 0.8 \
  --max-model-len 2048 \
  --max-lora-rank 64 \
  --enable-lora \
  --lora-modules arabic-chunker="{adapter_model_id}" > nohup.out 2>&1 &

print("Waiting for the server to start (this can take 3-5 minutes)...")
server_ready = False
for i in range(90):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("\nServer is ready.")
            server_ready = True
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(5)
    print(f"Waiting... ({(i+1)*5}s)", end="\r")

if not server_ready:
    print("\nServer did not respond within the timeout. Last log lines:")
    !tail -n 60 nohup.out

In [ ]:
!tail -n 60 nohup.out

# Inference

In [ ]:
# ============================================================
# Inference — single request smoke test before load testing
# ============================================================
from transformers import AutoTokenizer
import requests
import json

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

system_message = (
    "You are a professional multilingual NLP data parser.\n"
    "Split the provided text into meaningful, self-contained semantic chunks.\n"
    "Follow the Task and Output Schema to generate the Output JSON.\n"
    "Do not add any introduction or conclusion.\n"
    "Preserve the original text exactly and respect its language — "
    "do not summarize, translate, or paraphrase."
)

schema_str = json.dumps({
    "properties": {
        "original_text_length": {
            "description": "Total number of characters in the original text.",
            "title": "Original Text Length",
            "type": "integer"
        },
        "semantic_chunks": {
            "description": "List of sequential chunks from the original text. Each chunk is self-contained and preserves the original language(s).",
            "items": {"type": "string"},
            "minItems": 2,
            "title": "Semantic Chunks",
            "type": "array"
        }
    },
    "required": ["original_text_length", "semantic_chunks"],
    "title": "SemanticChunking",
    "type": "object"
}, ensure_ascii=False)

test_text = (
    "أعلنت شركة OpenAI عن إطلاق نموذجها الجديد في مطلع هذا العام. "
    "النموذج يدعم اللغة العربية بشكل كامل ويتفوق في مهام التلخيص والتصنيف. "
    "من جهة أخرى، أعلنت Google عن تحديثات مماثلة لنموذج Gemini الخاص بها."
)

test_messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": (
        f"# Input Text:\n{test_text}\n\n"
        f"# Task:\nSplit the provided text into meaningful, self-contained semantic chunks. "
        f"Do NOT summarize or leave out words.\n\n"
        f"# Output Schema:\n{schema_str}\n\n"
        f"# Output JSON:\n```json\n"
    )}
]

prompt = tokenizer.apply_chat_template(
    test_messages, tokenize=False, add_generation_prompt=True
)

vllm_model_id = "arabic-chunker"

try:
    llm_response = requests.post(
        "http://localhost:8000/v1/completions",
        json={
            "model": vllm_model_id,
            "prompt": prompt,
            "max_tokens": 1000,
            "temperature": 0.1
        },
        timeout=30
    )
    llm_response.raise_for_status()
    response_json = llm_response.json()

    print("=== Raw Response ===")
    print(response_json)

    raw_text = response_json["choices"][0]["text"].strip().removesuffix("```").strip()
    parsed = json.loads(raw_text)

    print("\n=== Parsed JSON ===")
    print(json.dumps(parsed, ensure_ascii=False, indent=2))

    chunks = parsed.get("semantic_chunks", [])
    print(f"\nNumber of chunks produced: {len(chunks)}")

except requests.exceptions.RequestException as e:
    print(f"Failed to connect to the server: {e}")
except (KeyError, json.JSONDecodeError) as e:
    print(f"Failed to parse the model response: {e}")
    print(f"Raw response: {response_json}")

# Load Testing

In [ ]:
# ============================================================
# Load Testing — setup
# ============================================================
import sys
import os

!{sys.executable} -m pip install -qU locust faker

if os.path.exists("/content/locustfile.py"):
    os.remove("/content/locustfile.py")

In [ ]:
%%writefile locustfile.py
import random
import json
from locust import HttpUser, task, between
from transformers import AutoTokenizer
from faker import Faker

fake = Faker('ar')
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-1.5B-Instruct")

system_message = "You are a professional multilingual NLP data parser.\nSplit the provided text into meaningful, self-contained semantic chunks.\nFollow the Task and Output Schema to generate the Output JSON.\nDo not add any introduction or conclusion.\nPreserve the original text exactly and respect its language — do not summarize, translate, or paraphrase."
schema_str = '{"properties": {"original_text_length": {"description": "Total number of characters in the original text.", "title": "Original Text Length", "type": "integer"}, "semantic_chunks": {"description": "List of sequential chunks from the original text. Each chunk is self-contained and preserves the original language(s).", "items": {"type": "string"}, "minItems": 2, "title": "Semantic Chunks", "type": "array"}}, "required": ["original_text_length", "semantic_chunks"], "title": "SemanticChunking", "type": "object"}'

class ChunkingLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def post_completion(self):
        model_id = "arabic-chunker"
        raw_text = fake.text(max_nb_chars=random.randint(400, 600))

        chunking_messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": f"# Input Text:\n{raw_text}\n\n# Task:\nSplit the provided text into meaningful, self-contained semantic chunks. Do NOT summarize or leave out words.\n\n# Output Schema:\n{schema_str}\n\n# Output JSON:\n```json\n"}
        ]

        prompt = tokenizer.apply_chat_template(
            chunking_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        message = {
            "model": model_id,
            "prompt": prompt,
            "max_tokens": 512,
            "temperature": 0.1
        }

        llm_response = self.client.post("/v1/completions", json=message)

        if llm_response.status_code == 200:
            with open("./vllm_tokens.txt", "a", encoding="utf-8") as dest:
                dest.write(json.dumps({
                    "prompt": prompt,
                    "response": llm_response.json()["choices"][0]["text"],
                }, ensure_ascii=False) + "\n")

In [ ]:
# ============================================================
# Load test — 20 concurrent users for 60 seconds
# ============================================================
import sys
import os

!rm -f vllm_tokens.txt && touch vllm_tokens.txt

!{sys.executable} -m locust --headless -f locustfile.py \
    --host=http://localhost:8000 \
    -u 20 -r 1 -t "60s" \
    --html=locust_results.html

from transformers import AutoTokenizer
import json

vllm_tokens = [
    json.loads(l) for l in open("./vllm_tokens.txt", encoding="utf-8") if l.strip() != ""
]

if vllm_tokens:
    in_tok  = sum(len(tokenizer.encode(r["prompt"])) for r in vllm_tokens)
    out_tok = sum(len(tokenizer.encode(r["response"])) for r in vllm_tokens)
    print(f"\nSuccessful requests: {len(vllm_tokens)}")
    print(f"Throughput: {(in_tok + out_tok) / 60:.2f} tokens/sec")
else:
    print("\nNo successful requests recorded.")
    !tail -n 60 nohup.out